## Basic Imports for Testing

In [1]:
import os
import finnhub
import pandas as pd
import json
import duckdb
from functools import reduce
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
api_key = os.getenv('FINNHUB_API_KEY')

# Setup client
finnhub_client = finnhub.Client(api_key=api_key)
test_symbol = 'AAPL'
test_symbol_name = 'apple'
test_symbol_list = ['AAPL', 'TSLA']
test_symbol_list_names = ['apple', 'tesla']

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

def normalize_json_dataframe(df):
    """
    Iterates through each row in the DataFrame, normalizes JSON content in each column,
    and creates new DataFrames from the normalized JSON, associating each resulting row
    with the original index value.
    
    Assumes each cell contains a JSON string, dict, or list of dicts.
    Returns a list of DataFrames, one per original row.
    """
    result_dataframes = {}
    
    for idx, row in df.iterrows():
        row_normalized_dfs = []
        
        for col in df.columns:
            cell_value = row[col]
            
            # Parse JSON if it's a string
            if isinstance(cell_value, str):
                try:
                    json_data = json.loads(cell_value)
                except json.JSONDecodeError:
                    continue  # Skip invalid JSON
            else:
                json_data = cell_value
            
            # Normalize if it's a dict or list of dicts
            if isinstance(json_data, (dict, list)):
                try:
                    normalized_df = pd.json_normalize(json_data)
                    normalized_df['original_index'] = idx  # Add original index to the normalized DataFrame
                    rename_cols = [i for i in normalized_df.columns if i == 'v']
                    for i in rename_cols:
                        normalized_df.rename(columns={i: f"{col}_{i}"}, inplace=True)
                    row_normalized_dfs.append(normalized_df)
                except Exception:
                    continue  # Skip if normalization fails
        
        # Combine all normalized DataFrames for this row
        if row_normalized_dfs:
            combined_df = reduce(lambda left, right: pd.merge(left, right, on=['period', 'original_index'], how='outer'), row_normalized_dfs)
            ## combined_df.set_index('original_index', inplace=True)
            result_dataframes[idx] = combined_df
    
    return result_dataframes

## Free Plan Endpoints

##### Company Basic Financials

In [ ]:
# Basic financials
## print(finnhub_client.company_basic_financials(test_symbol, 'all'))
df = pd.DataFrame(finnhub_client.company_basic_financials(test_symbol, 'all'))
annual_and_quarterly_pre_df = df.copy(deep=True)
annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['annual']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
annual_df = pd.json_normalize(annual_pre_df['series'])
quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['quarterly']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
quarterly_df = pd.json_normalize(quarterly_pre_df['series'])
df = df.drop(['annual','quarterly'], axis=0)
df = df.drop(columns=['metricType','series']).reset_index()
df

In [ ]:
duckdb.register("df_view", df)
res = duckdb.query("SELECT * FROM df_view WHERE index == '10DayAverageTradingVolume'").to_df()
res

In [ ]:
symbol_name = test_symbol
annual_json_parsed_dfs = normalize_json_dataframe(annual_df)
duckdb.register("annual_json_parsed_df_view", annual_json_parsed_dfs[symbol_name])
res = duckdb.query("SELECT * FROM annual_json_parsed_df_view WHERE period == '1985-09-27'").to_df()
res


In [ ]:
symbol_name = test_symbol
quarterly_json_parsed_dfs = normalize_json_dataframe(quarterly_df)
duckdb.register("quarterly_json_parsed_df_view", quarterly_json_parsed_dfs[symbol_name])
res = duckdb.query("SELECT * FROM quarterly_json_parsed_df_view WHERE period == '1989-06-30'").to_df()
res

##### Earnings Surprises

In [ ]:
# Earnings surprises
## print(finnhub_client.company_earnings(test_symbol, limit=5))
df = pd.DataFrame(finnhub_client.company_earnings(test_symbol))
df

##### Company News

In [ ]:
# Company News
# Need to use _from instead of from to avoid conflict
df = pd.json_normalize(finnhub_client.company_news(test_symbol, _from=pd.to_datetime("today").date(), to=pd.to_datetime("today").date()))
df['datetime_converted'] = pd.to_datetime(df['datetime'], unit='s')
df.drop(columns=['image'], inplace=True)
df

##### Company Peers

In [ ]:
# Company Peers
df = pd.DataFrame(finnhub_client.company_peers(test_symbol))
df['symbol'] = test_symbol
df = df[df[0]!=df['symbol']]
df

##### Company Profile 2

In [ ]:
# Company Profile 2
df = pd.json_normalize(finnhub_client.company_profile2(symbol=test_symbol))
df

##### List Country

In [ ]:
# List country
df = pd.json_normalize(finnhub_client.country())
df

##### Crypto Exchange

In [ ]:
# Crypto Exchange
df = pd.DataFrame(finnhub_client.crypto_exchanges())
df

##### Crypto Symbols

In [ ]:
# Crypto symbols
df = pd.json_normalize(finnhub_client.crypto_symbols('BINANCE'))
df

##### Financials Reported

In [9]:
# Financials as reported
df = pd.json_normalize(finnhub_client.financials_reported(symbol=test_symbol, freq='annual'))
df = normalize_json_dataframe(pd.DataFrame(df['data']))
df = pd.DataFrame(df[0])
df

,accessNumber,symbol,cik,year,quarter,form,startDate,endDate,filedDate,acceptedDate,report.bs,report.ic,report.cf,original_index
0,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
1,0000320193-24-000123,AAPL,320193,2024,0,10-K,2023-10-01 00:00:00,2024-09-28 00:00:00,2024-11-01 00:00:00,2024-11-01 06:01:36,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
2,0000320193-23-000106,AAPL,320193,2023,0,10-K,2022-09-25 00:00:00,2023-09-30 00:00:00,2023-11-03 00:00:00,2023-11-02 18:08:27,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
3,0000320193-22-000108,AAPL,320193,2022,0,10-K,2021-09-26 00:00:00,2022-09-24 00:00:00,2022-10-28 00:00:00,2022-10-27 18:01:14,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
4,0000320193-21-000105,AAPL,320193,2021,0,10-K,2020-09-27 00:00:00,2021-09-25 00:00:00,2021-10-29 00:00:00,2021-10-28 18:04:28,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
5,0000320193-20-000096,AAPL,320193,2020,0,10-K,2019-09-29 00:00:00,2020-09-26 00:00:00,2020-10-30 00:00:00,2020-10-29 18:06:25,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
6,0000320193-19-000119,AAPL,320193,2019,0,10-K,2018-09-30 00:00:00,2019-09-28 00:00:00,2019-10-31 00:00:00,2019-10-30 18:12:36,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ...",0
7,0000320193-18-000145,AAPL,320193,2018,0,10-K,2017-10-01 00:00:00,2018-09-29 00:00:00,2018-11-05 00:00:00,2018-11-05 08:01:40,"[{'concept': 'us-gaap_AccountsPayableCurrent',...",[{'concept': 'us-gaap_ComprehensiveIncomeNetOf...,[{'concept': 'us-gaap_CashAndCashEquivalentsPe...,0
8,0000320193-17-000070,AAPL,320193,2017,0,10-K,2016-09-25 00:00:00,2017-09-30 00:00:00,2017-11-03 00:00:00,2017-11-03 08:01:37,"[{'concept': 'us-gaap_AccountsPayableCurrent',...",[{'concept': 'us-gaap_CommonStockDividendsPerS...,[{'concept': 'us-gaap_CashAndCashEquivalentsPe...,0
9,0001628280-16-020309,AAPL,320193,2016,0,10-K,2015-09-27 00:00:00,2016-09-24 00:00:00,2016-10-26 00:00:00,2016-10-26 16:42:16,"[{'concept': 'us-gaap_AccountsPayableCurrent',...",[{'concept': 'us-gaap_CommonStockDividendsPerS...,[{'concept': 'aapl_PaymentsOfDividendsAndDivid...,0


In [ ]:
# Forex exchanges
print(finnhub_client.forex_exchanges())

In [ ]:
# Forex symbols
print(finnhub_client.forex_symbols('OANDA'))

In [ ]:
# General news
print(finnhub_client.general_news('forex', min_id=0))

In [ ]:
# IPO calendar
print(finnhub_client.ipo_calendar(_from="2020-05-01", to="2020-06-01"))

In [ ]:
# Quote
print(finnhub_client.quote(test_symbol))

In [ ]:
# Recommendation trends
print(finnhub_client.recommendation_trends(test_symbol))

In [ ]:
# Stock symbols
print(finnhub_client.stock_symbols('US')[0:5])

In [ ]:
# Earnings Calendar
print(finnhub_client.earnings_calendar(_from="2026-03-16", to="2026-03-20", symbol=test_symbol, international=False))

In [ ]:
# Covid-19
print(finnhub_client.covid19())

In [ ]:
# FDA Calendar
print(finnhub_client.fda_calendar())

In [ ]:
# Symbol lookup
print(finnhub_client.symbol_lookup(test_symbol_name))

In [ ]:
# Visa application
print(finnhub_client.stock_visa_application(test_symbol, "2021-01-01", "2022-06-15"))

In [ ]:
# Insider sentiment
print(finnhub_client.stock_insider_sentiment(test_symbol, '2021-01-01', '2022-03-01'))

In [ ]:
# USA Spending
print(finnhub_client.stock_usa_spending(test_symbol, "2021-01-01", "2022-06-15"))

In [ ]:
## Market Holday / Status
print(finnhub_client.market_holiday(exchange='US'))
print(finnhub_client.market_status(exchange='US'))

## Paid Endpoints